## Homework 8: API Client
Kaiyi Wang - kw3297
# Part 1: The API
I chose NPS API to collect information about parks and visitor centers. This assignment uses requests directly.
# a) Python package
- The Python wrapper is Python-natlparks. Version 0.1.6, released March 16, 2025.
# b) Authentication
- A free API Key was requested through the NPS registration form. Loaded into the .envrc file. I will send it using the X-Api-Key request header.
Registration: https://www.nps.gov/subjects/developer/get-started.htm
Authentication Guide: https://www.nps.gov/subjects/developer/guides.htm
# c) ENdpoints and pagination
- parks: park names, descriptions, locations, and other park information.
- visitorcenters: visitor center locations, contact information, and operating hours.
Use offset pagination. The limit parameter controls the number of results requested, and start specifies the zero-based starting position. For example 50 records per request, 0, 50, 100.
# d) Rate Limits
- 1,000 requests per hour per API key. The X-RateLimit-Limit and X-RateLimit-Remaining headers report the limit and remaining requests.

## Part 2: Load the API Key Safely.

In [1]:
import os
import requests
import pandas as pd

# API Key loaded
api_key = os.getenv("NPS_API_KEY")
print("API key found:", bool(api_key))

API key found: True


## Part 3: Request
I will request the first five park records. The limit parameter sets the number of records requested, and start=0 begins with the first record. The API key is sent in the request header, and the timeout is 30 sec. The records are in data{"data"]. The top-level fields total, limit, and start contain pagination information.

In [2]:
url = "https://developer.nps.gov/api/v1/parks"
headers = {"X-Api-Key": api_key}
params = {"limit": 5, "start": 0}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

response.raise_for_status()
data = response.json()

# Response structure and pagination information.
print("Top-level keys:", list(data.keys()))
print("Total records:", data["total"])
print("Starting position:", data["start"])
print("Limit:", data["limit"])
print("Records returned:", len(data["data"]))

# Display a small sample
for park in data["data"][:2]:
    print(park["fullName"], "-", park["states"])

Status code: 200
Content-Type: application/json;charset=utf-8
Top-level keys: ['total', 'limit', 'start', 'data']
Total records: 475
Starting position: 0
Limit: 5
Records returned: 5
Abraham Lincoln Birthplace National Historical Park - KY
Acadia National Park - ME


## Part 4: Collect and Clean the Dataset

### a. Collect all pages


I request 100 records per page and increase the starting position after
each request. The loop stops when no records remain, the rate limit is
exhausted, or the safety limit of 10 pages is reached.

In [3]:
params = {"limit": 100, "start": 0}
all_parks = []
max_pages = 10
page = 0

while page < max_pages:
    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )
    response.raise_for_status()
    data = response.json()

    parks = data["data"]
    total = int(data["total"])

    # Check how many requests remain.
    remaining = response.headers.get("X-RateLimit-Remaining")
    print("Requests remaining:", remaining)

    # An empty page means there are no more records.
    if not parks:
        break

    all_parks.extend(parks)
    page += 1
    print(f"Page {page}: collected {len(all_parks)} records")

    params["start"] += params["limit"]

    if remaining is not None and int(remaining) == 0:
        print("Rate limit reached. Stopping.")
        break

print("Records collected:", len(all_parks))
print("API total:", total)
print("Count matches:", len(all_parks) == total)

Requests remaining: 984
Page 1: collected 100 records
Requests remaining: 984
Page 2: collected 200 records
Requests remaining: 982
Page 3: collected 300 records
Requests remaining: 981
Page 4: collected 400 records
Requests remaining: 981
Page 5: collected 475 records
Requests remaining: 979
Records collected: 475
API total: 475
Count matches: True


### Explanation
The 475 collected records match the total reported by the API.
I select identifying information, descriptions, states, and coordinates.
Latitude and longitude are converted from text to numeric values;
blank or invalid coordinates become missing values.

In [4]:
# Select useful columns from the park records.
columns = [
    "parkCode", "fullName", "designation", "states",
    "description", "latitude", "longitude", "url"
]
df = pd.DataFrame(all_parks)[columns].copy()

# Convert coordinates from strings to numbers.
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

df.head()

,parkCode,fullName,designation,states,description,latitude,longitude,url
0,abli,Abraham Lincoln Birthplace National Historical...,National Historical Park,KY,For over a century people from around the worl...,37.585866,-85.673305,https://www.nps.gov/abli/index.htm
1,acad,Acadia National Park,National Park,ME,Acadia National Park protects the natural beau...,44.409286,-68.247501,https://www.nps.gov/acad/index.htm
2,adam,Adams National Historical Park,National Historical Park,MA,From the sweet little farm at the foot of Penn...,42.255396,-71.011604,https://www.nps.gov/adam/index.htm
3,afam,African American Civil War Memorial,,DC,"Over 200,000 African-American soldiers and sai...",38.916600,-77.026000,https://www.nps.gov/afam/index.htm
4,afbg,African Burial Ground National Monument,National Monument,NY,The African Burial Ground stands as the oldest...,40.714527,-74.004474,https://www.nps.gov/afbg/index.htm


In [5]:
print("\nMissing values:")
# Treat empty strings as missing values.
df = df.replace("", pd.NA)
df.info()
print(df.isnull().sum())
# Save the dataset without the DataFrame index.
df.to_csv("data.csv", index=False)
df.describe()


Missing values:
<class 'pandas.DataFrame'>
RangeIndex: 475 entries, 0 to 474
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   parkCode     475 non-null    str    
 1   fullName     475 non-null    str    
 2   designation  440 non-null    str    
 3   states       475 non-null    str    
 4   description  471 non-null    str    
 5   latitude     475 non-null    float64
 6   longitude    475 non-null    float64
 7   url          475 non-null    str    
dtypes: float64(2), str(6)
memory usage: 29.8 KB
parkCode        0
fullName        0
designation    35
states          0
description     4
latitude        0
longitude       0
url             0
dtype: int64


,latitude,longitude
count,475.000000,475.000000
mean,38.551601,-94.094235
std,7.846038,27.028049
min,-14.228659,-169.850378
25%,35.379745,-109.396369
50%,38.877313,-88.710010
75%,41.658112,-77.035766
max,71.298324,145.723110


The dataset contains 475 National Park Service records and eight columns describing their identifiers, names, designations, states, descriptions, coordinates, and URLs. Latitude and longitude were converted to numeric values, and empty strings were treated as missing values. Includes national parks, historical parks, and memorials.

## Part 5: Create Function

Wrap the previous code into a function, using pagination, and return a cleaned DataFrame. It can customize the state filter, page size, maximum number of pages, and API key. Key from the environment by default.

In [6]:
def get_parks(state_code="", limit=100, max_pages=10, api_key=None):
    """Retrieve NPS park records as a cleaned DataFrame.

    Parameters
    ----------
    state_code : str
        State abbreviation, such as "NY". Empty means all states.
    limit : int
        Number of records requested per page.
    max_pages : int
        Maximum number of pages to request.
    api_key : str or None
        API key. If None, read NPS_API_KEY from the environment.

    Returns
    -------
    pandas.DataFrame
        Park information with numeric coordinates and missing values.
        Results may be incomplete if max_pages is reached.

    Example
    -------
    parks = get_parks(state_code="NY")
    """
    if api_key is None:
        api_key = os.getenv("NPS_API_KEY")

    if not api_key:
        raise ValueError("No API key found. Set NPS_API_KEY first.")

    url = "https://developer.nps.gov/api/v1/parks"
    headers = {"X-Api-Key": api_key}
    params = {"limit": limit, "start": 0}

    if state_code:
        params["stateCode"] = state_code

    all_parks = []
    page = 0

    while page < max_pages:
        try:
            response = requests.get(
                url,
                headers=headers,
                params=params,
                timeout=30
            )
            response.raise_for_status()

        except requests.exceptions.RequestException as error:
            # Report the problem without displaying the key or headers.
            if error.response is not None:
                status = error.response.status_code
                if status in (401, 403):
                    print("Authentication failed. Check your API key.")
                elif status == 429:
                    print("Too many requests. Wait before trying again.")
                else:
                    print(f"Request failed with status {status}.")
            else:
                print("Network error or timeout. Try again later.")
            raise RuntimeError("Could not retrieve park data.") from None

        data = response.json()
        parks = data["data"]

        if not parks:
            break

        all_parks.extend(parks)
        page += 1
        params["start"] += limit

        remaining = response.headers.get("X-RateLimit-Remaining")
        if remaining is not None and int(remaining) == 0:
            print("Rate limit reached. Returning records collected so far.")
            break

    # Keep the same columns and cleaning steps used in Part 4.
    columns = [
        "parkCode", "fullName", "designation", "states",
        "description", "latitude", "longitude", "url"
    ]
    result = pd.DataFrame(all_parks, columns=columns)
    result = result.replace("", pd.NA)
    result["latitude"] = pd.to_numeric(result["latitude"], errors="coerce")
    result["longitude"] = pd.to_numeric(result["longitude"], errors="coerce")

    return result

In [7]:
# Retrieve all parks using the default settings.
parks_df = get_parks()

print("Rows and columns:", parks_df.shape)
parks_df.head()

Rows and columns: (475, 8)


,parkCode,fullName,designation,states,description,latitude,longitude,url
0,abli,Abraham Lincoln Birthplace National Historical...,National Historical Park,KY,For over a century people from around the worl...,37.585866,-85.673305,https://www.nps.gov/abli/index.htm
1,acad,Acadia National Park,National Park,ME,Acadia National Park protects the natural beau...,44.409286,-68.247501,https://www.nps.gov/acad/index.htm
2,adam,Adams National Historical Park,National Historical Park,MA,From the sweet little farm at the foot of Penn...,42.255396,-71.011604,https://www.nps.gov/adam/index.htm
3,afam,African American Civil War Memorial,NaN,DC,"Over 200,000 African-American soldiers and sai...",38.916600,-77.026000,https://www.nps.gov/afam/index.htm
4,afbg,African Burial Ground National Monument,National Monument,NY,The African Burial Ground stands as the oldest...,40.714527,-74.004474,https://www.nps.gov/afbg/index.htm


### Test Invalid API

In [8]:
# Use a deliberately invalid key.
try:
    get_parks(api_key="invalid_key")
except RuntimeError as error:
    print(error)

Authentication failed. Check your API key.
Could not retrieve park data.


### API Client Class

The client uses requests.Session to reuse the API-key header across
requests. Shared methods handle requests and pagination, while
get_parks and get_visitor_centers retrieve data from different endpoints.

In [9]:
class NPSClient:
    """Retrieve parks and visitor centers using a shared session."""

    def __init__(self, api_key=None):
        if api_key is None:
            api_key = os.getenv("NPS_API_KEY")

        if not api_key:
            raise ValueError("No API key found. Set NPS_API_KEY first.")

        self.base_url = "https://developer.nps.gov/api/v1"
        self.session = requests.Session()
        self.session.headers.update({"X-Api-Key": api_key})

    def _make_request(self, method, endpoint, **kwargs):
        """Send a request and report errors without exposing the key."""
        url = f"{self.base_url}/{endpoint}"

        try:
            response = self.session.request(
                method, url, timeout=30, **kwargs
            )
            response.raise_for_status()
            return response

        except requests.exceptions.RequestException as error:
            if error.response is not None:
                status = error.response.status_code
                if status in (401, 403):
                    print("Authentication failed. Check your API key.")
                elif status == 429:
                    print("Too many requests. Wait before trying again.")
                else:
                    print(f"Request failed with status {status}.")
            else:
                print("Network error or timeout. Try again later.")

            raise RuntimeError("Could not retrieve NPS data.") from None

    def _paginate(self, endpoint, params=None, max_pages=10):
        """Collect records until an empty page or a safety limit."""
        params = params or {}
        params["limit"] = 100
        params["start"] = 0
        results = []
        page = 0

        while page < max_pages:
            response = self._make_request(
                "GET", endpoint, params=params
            )
            data = response.json()
            records = data["data"]

            if not records:
                break

            results.extend(records)
            page += 1
            params["start"] += params["limit"]

            remaining = response.headers.get("X-RateLimit-Remaining")
            if remaining is not None and int(remaining) == 0:
                print("Rate limit reached. Returning collected records.")
                break

        return results

    def get_parks(self, state_code="", max_pages=10):
        """Return a cleaned parks DataFrame, optionally filtered by state."""
        params = {}
        if state_code:
            params["stateCode"] = state_code

        records = self._paginate("parks", params, max_pages)
        columns = [
            "parkCode", "fullName", "designation", "states",
            "description", "latitude", "longitude", "url"
        ]
        result = pd.DataFrame(records, columns=columns)
        result = result.replace("", pd.NA)
        result["latitude"] = pd.to_numeric(
            result["latitude"], errors="coerce"
        )
        result["longitude"] = pd.to_numeric(
            result["longitude"], errors="coerce"
        )
        return result

    def get_visitor_centers(self, park_code="", max_pages=10):
        """Return visitor centers, optionally filtered by park code."""
        params = {}
        if park_code:
            params["parkCode"] = park_code

        records = self._paginate("visitorcenters", params, max_pages)
        columns = [
            "id", "name", "parkCode", "description",
            "latitude", "longitude", "url"
        ]
        result = pd.DataFrame(records, columns=columns)
        result = result.replace("", pd.NA)
        result["latitude"] = pd.to_numeric(
            result["latitude"], errors="coerce"
        )
        result["longitude"] = pd.to_numeric(
            result["longitude"], errors="coerce"
        )
        return result

In [10]:
client = NPSClient()

ny_parks = client.get_parks(state_code="NY")
print("New York parks:", len(ny_parks))
ny_parks.head()

New York parks: 32


,parkCode,fullName,designation,states,description,latitude,longitude,url
0,afbg,African Burial Ground National Monument,National Monument,NY,The African Burial Ground stands as the oldest...,40.714527,-74.004474,https://www.nps.gov/afbg/index.htm
1,appa,Appalachian National Scenic Trail,National Scenic Trail,"CT,GA,MA,MD,ME,NC,NH,NJ,NY,PA,TN,VA,VT,WV","The Appalachian Trail is a 2,190+ mile long pu...",40.410296,-76.433755,https://www.nps.gov/appa/index.htm
2,cajo,Captain John Smith Chesapeake National Histori...,National Historic Trail,"VA,MD,DE,DC,PA,NY",People first arrived in the Chesapeake Bay dur...,38.971601,-76.483355,https://www.nps.gov/cajo/index.htm
3,cacl,Castle Clinton National Monument,National Monument,NY,"Located at the southern tip of Manhattan, Cast...",40.703478,-74.016806,https://www.nps.gov/cacl/index.htm
4,cbpo,Chesapeake Bay,NaN,"DC,DE,MD,NY,PA,VA,WV",Over 41 million acres are connected through th...,38.971601,-76.483355,https://www.nps.gov/cbpo/index.htm


In [11]:
acadia_centers = client.get_visitor_centers(park_code="acad")
print("Acadia visitor centers:", len(acadia_centers))
acadia_centers.head()

Acadia visitor centers: 6


,id,name,parkCode,description,latitude,longitude,url
0,2DC6EAAC-3718-4438-9630-85E55A3DA496,Hulls Cove Visitor Center,acad,"Located close to Bar Harbor off Route 3, the H...",44.408542,-68.246137,https://www.nps.gov/acad/planyourvisit/hours.htm
1,CA752C71-AFF4-4599-AE84-DEDFE0A21EAA,Sieur de Monts Nature Center,acad,Sieur de Monts Nature Center has visitor servi...,44.362109,-68.207848,http://www.nps.gov/acad/planyourvisit/hours.htm
2,99B33FA9-2579-415C-B2C7-2A29879744F8,Acadia Gateway Center,acad,The Acadia Gateway Center is an intermodal tra...,44.470549,-68.366182,https://www.mainetourism.com/listing/maine-sta...
3,109BDB6B-C627-4C8D-87DA-A4F687E81BCE,Islesford Historical Museum,acad,The Islesford Historical Museum continues its ...,44.261592,-68.239818,http://www.nps.gov/acad/planyourvisit/hours.htm
4,46CA9FEA-036F-42CC-9B75-48F0BF531F7F,Schoodic Institute Welcome Center at Rockefell...,acad,The Schoodic Institute Welcome Center at Rocke...,44.336076,-68.059139,http://www.nps.gov/acad/planyourvisit/hours.htm
